# v10 学習・CV比較 ― 逆方向PF特徴の効果検証(LightGBM)

`artifacts_v10/data/train.parquet` ができている前提。

3つの特徴量セットで CV(GroupKFold by well)を回し、**overall RMSE** を比較する:

1. **順方向のみ**(=v9 と同じ特徴。`bpf_` / `cmp_` を除外)
2. **順方向 + 逆方向**(+ `bpf_*`)
3. **順方向 + 逆方向 + 比較**(+ `cmp_*`)

評価方針(CLAUDE.md):
- Kaggle は overall RMSE(全行プール)。本NBも OOF 全行をプールして RMSE を出す。
- `target = TVT - last_tvt`(坑井定数オフセット)なので target上RMSE = TVT上RMSE。
- 参考に「坑井ごとRMSEの平均」も出す(overall とは別物)。

注意: 逆方向特徴は seed ごとに列を持たせない(parquet 側で 32本をアンサンブル済み)。
CV はリーク防止のため必ず **well 単位のグループ分割**。

In [1]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error

In [ ]:
# ============================== CONFIG ==============================
PROJECT_DIR = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v10")
PARQUET     = PROJECT_DIR / "v10" / "artifacts_v10_2" / "data" / "train.parquet"
OUT_DIR     = PROJECT_DIR / "v10" / "artifacts_v10_2"
N_SPLITS    = 5
SEED        = 0
DEVICE_TYPE = "cpu"   # GPU対応 LightGBM があれば "gpu" に。pip 既定wheelは CPU のみのことが多い。
SAVE_OOF    = True

# v9 の主 LightGBM パラメータに準拠(モデルは3実験で固定)
LGB_PARAMS = dict(
    boosting_type="gbdt", num_leaves=255, min_child_samples=15,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    reg_lambda=3.0, reg_alpha=0.05, objective="regression",
    verbose=-1, n_jobs=-1, max_bin=255,
    learning_rate=0.030, n_estimators=5000, seed=123,
)
EARLY_STOP   = 200
NON_FEATURES = {"well", "id", "target"}   # well=グループ, id=提出ID(文字列), target=目的変数

In [3]:
def select_features(cols, mode):
    """mode: 'fwd' / 'fwd_bwd' / 'fwd_bwd_cmp'"""
    feats = [c for c in cols if c not in NON_FEATURES]
    if mode == "fwd":
        feats = [c for c in feats if not c.startswith("bpf_") and not c.startswith("cmp_")]
    elif mode == "fwd_bwd":
        feats = [c for c in feats if not c.startswith("cmp_")]
    elif mode == "fwd_bwd_cmp":
        pass
    else:
        raise ValueError(mode)
    return feats

In [4]:
# データ読み込み
df = pd.read_parquet(PARQUET)
assert "target" in df.columns and "well" in df.columns, "parquet に target / well 列が必要"
groups = df["well"]
print(f"rows={len(df)}  wells={groups.nunique()}  cols={df.shape[1]}")
for mode in ["fwd", "fwd_bwd", "fwd_bwd_cmp"]:
    print(f"  {mode:14s}: {len(select_features(df.columns, mode))} features")
df.head()

rows=3783989  wells=773  cols=658
  fwd           : 511 features
  fwd_bwd       : 627 features
  fwd_bwd_cmp   : 655 features


,well,id,last_known_tvt,beam_gr_d,beam_hard_d,beam_loose_d,beam_mid_d,beam_smooth_d,beam_best_d,beam_mean_d,...,bpf_3_rate_rmean21,bpf_3_rate_rstd21,bpf_3_acc_rmean21,cmp_fwd_bwd_diff_3,cmp_fwd_bwd_absdiff_3,cmp_fwd_bwd_mean_3,cmp_fwd_bwd_rate_diff_3,cmp_fwd_bwd_absdiff_rmean21_3,cmp_fwd_bwd_grabsmin_diff_3,cmp_fwd_bwd_std_diff_3
0,000d7d20,000d7d20_1442,11747.370117,1.080078,0.080078,1.080078,1.080078,1.080078,0.080078,0.746745,...,0.038663,0.007301,-0.000688,-5.408203,5.408203,11750.116211,0.006836,5.446378,-2.834648,-4.292952
1,000d7d20,000d7d20_1443,11747.370117,1.080078,0.080078,1.080078,1.080078,1.080078,0.080078,0.746745,...,0.037476,0.008086,0.000122,-5.401367,5.401367,11750.154297,0.002441,5.449544,-5.067772,-4.288408
2,000d7d20,000d7d20_1444,11747.370117,1.080078,0.080078,1.080078,1.080078,1.080078,0.080078,0.746745,...,0.038199,0.008168,0.002009,-5.403320,5.403320,11750.187500,-0.006348,5.453576,-5.841133,-4.279451
3,000d7d20,000d7d20_1445,11747.370117,1.080078,0.080078,1.080078,1.080078,1.080078,0.080078,0.746745,...,0.040737,0.012320,0.003557,-5.414062,5.414062,11750.217773,-0.008301,5.461147,-4.211220,-4.269049
4,000d7d20,000d7d20_1446,11747.370117,1.080078,0.080078,1.080078,1.080078,1.080078,0.080078,0.746745,...,0.044303,0.018214,0.004329,-5.419922,5.419922,11750.249023,-0.008301,5.472982,-1.379456,-4.261939


In [5]:
def run_cv(df, features, target, groups, n_splits, lgb_params, early_stop, device_type, seed):
    """GroupKFold CV。OOF予測・overall RMSE・坑井平均RMSE・fold別RMSE を返す。"""
    X = df[features]
    y = df[target].to_numpy(np.float32)
    oof = np.full(len(df), np.nan, np.float32)
    fold_rmse = []
    gkf = GroupKFold(n_splits=n_splits)
    for k, (tr, va) in enumerate(gkf.split(X, y, groups)):
        params = dict(lgb_params); params["device_type"] = device_type
        model = LGBMRegressor(**params)
        model.fit(
            X.iloc[tr], y[tr],
            eval_set=[(X.iloc[va], y[va])],
            eval_metric="rmse",
            callbacks=[early_stopping(early_stop, verbose=False), log_evaluation(0)],
        )
        pred = model.predict(X.iloc[va], num_iteration=model.best_iteration_)
        oof[va] = pred.astype(np.float32)
        r = root_mean_squared_error(y[va], pred)
        fold_rmse.append(float(r))
        print(f"    fold {k}: rmse={r:.4f}  best_iter={model.best_iteration_}")
    overall = float(root_mean_squared_error(y, oof))
    tmp = pd.DataFrame({"well": groups.to_numpy(), "e2": (oof - y) ** 2})
    per_well = float(np.sqrt(tmp.groupby("well")["e2"].mean()).mean())
    return dict(oof=oof, overall=overall, per_well_mean=per_well, fold_rmse=fold_rmse)

In [ ]:
# 3実験を実行
experiments = [
    # ("(1) 順方向のみ(=v9)",   "fwd"),
    ("(2) 順方向+逆方向",      "fwd_bwd"),
    # ("(3) 順方向+逆方向+比較", "fwd_bwd_cmp"),
]
results = {}
for label, mode in experiments:
    feats = select_features(df.columns, mode)
    print(f"\n===== {label}  (features={len(feats)}) =====")
    t0 = time.perf_counter()
    res = run_cv(df, feats, "target", groups, N_SPLITS, LGB_PARAMS, EARLY_STOP, DEVICE_TYPE, SEED)
    res.update(label=label, mode=mode, n_features=len(feats), time=time.perf_counter()-t0)
    results[mode] = res
    print(f"  overall RMSE = {res['overall']:.4f}   (坑井平均RMSE={res['per_well_mean']:.4f}, {res['time']:.1f}s)")
    if SAVE_OOF:
        np.save(OUT_DIR / f"oof_{mode}.npy", res["oof"])


===== (1) 順方向のみ(=v9)  (features=511) =====
    fold 0: rmse=7.3448  best_iter=417
    fold 1: rmse=7.4263  best_iter=185
    fold 2: rmse=7.3375  best_iter=266
    fold 3: rmse=7.7635  best_iter=153
    fold 4: rmse=9.8048  best_iter=957
  overall RMSE = 7.9914   (坑井平均RMSE=6.0911, 1704.2s)

===== (2) 順方向+逆方向  (features=627) =====
    fold 0: rmse=6.8253  best_iter=1061
    fold 1: rmse=6.9773  best_iter=702
    fold 2: rmse=7.0656  best_iter=1245
    fold 3: rmse=7.4505  best_iter=94
    fold 4: rmse=9.1990  best_iter=667
  overall RMSE = 7.5537   (坑井平均RMSE=5.7349, 3084.0s)


In [7]:
# 比較サマリ
base = results["fwd"]["overall"]
rows = []
for _, mode in experiments:
    r = results[mode]
    rows.append(dict(experiment=r["label"], n_feat=r["n_features"],
                     overall=round(r["overall"], 4),
                     per_well=round(r["per_well_mean"], 4),
                     delta_vs_fwd=round(r["overall"] - base, 4)))
summary = pd.DataFrame(rows)
print("delta_vs_fwd: (1)順方向のみ との overall RMSE 差。マイナスなら逆方向特徴で改善。")
summary

delta_vs_fwd: (1)順方向のみ との overall RMSE 差。マイナスなら逆方向特徴で改善。


,experiment,n_feat,overall,per_well,delta_vs_fwd
0,(1) 順方向のみ(=v9),511,7.9914,6.0911,0.0000
1,(2) 順方向+逆方向,627,7.5537,5.7349,-0.4377
